# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the available record sets in the dataset. Each record set, field, and column will be referenced by its `@id`.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '-')}")
    print(f"  Description: {rs.get('description', '-')}")
    # List fields
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    Field @id: {field.get('@id', field)} | Name: {field.get('name', '-')}")
            else:
                print(f"    Field @id: {field}")
    print()

# For example, to view a sample record from the first record set:
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"First record from RecordSet {first_rs_id}:")
    for rec in dataset.records(record_set=first_rs_id):
        print(json.dumps(rec, indent=2))
        break

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. We use the `@id` fields as references.

In [ ]:
# List all record set `@id`s for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for RecordSet @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    else:
        print(f"\nNo records found for RecordSet @id: {record_set_id}")

# For demonstration, select the first available DataFrame
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    print(f"\nUsing example DataFrame for RecordSet @id: {example_record_set_id}")
    print(df.head())
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
import numpy as np

# EDA only if a DataFrame is available
if example_record_set_id:
    df = dataframes[example_record_set_id]
    print(f"Columns in the example DataFrame:")
    print(df.columns.tolist())

    # Try to auto-detect a numeric field
    numeric_field = None
    for col in df.columns:
        # Try converting to numeric
        try:
            series_numeric = pd.to_numeric(df[col], errors='coerce')
            # Heuristic: Pick a field with at least 5 unique numeric non-NA values
            if series_numeric.notna().sum() > 5:
                numeric_field = col
                break
        except:
            pass
    print(f"\nSelected numeric field for analysis: {numeric_field}")

    if numeric_field is not None:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to pick a categorical/group field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
                if df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col]):
                    group_field = col
                    break
        print(f"\nGrouping by field: {group_field}")

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No data available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The example below visualizes the distribution of the numeric field if available:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and process the FAIR^2 dataset:
- Inspected dataset metadata and structure.
- Extracted record sets and loaded sample data.
- Performed basic exploratory data analysis (EDA) on selected fields.
- Created basic visualizations of the data.

Further analysis can be conducted according to specific research questions or data science tasks.